# Baseline: Replicating Figure 1 (FOMC Fed Funds Rate Forecast Errors)

Reproduces the methodology behind Figure 1 of Diercks, Katz and Wright (2026), *"Kalshi and the Rise of Macro Markets"*: mean absolute forecast error of the fed funds rate, plotted against the number of days before each FOMC meeting, comparing Kalshi's implied mean/median/mode to fed funds futures.

**This notebook ships with a synthetic data generator so it runs end to end with no credentials.** Swap `generate_synthetic_kalshi_ladder(...)` for a real pull via `src/kalshi_api.py` once you're ready to reproduce the figure on actual trade data -- see `paper/sections/2_replication.md` for the data-sourcing checklist (Kalshi trades, SME PDFs, fed funds futures settlement, Bloomberg consensus).

In [ ]:
import sys
sys.path.append("../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from kalshi_utils import ladder_to_pdf

rng = np.random.default_rng(7)

## 1. Synthetic data generator

Simulates, for one FOMC meeting, a daily-updating strike ladder of "Yes" prices over the 160 days before the meeting. The ladder converges toward the true outcome as the meeting approaches, with noise that shrinks over time -- mimicking the FEDS paper's qualitative description of forecast errors shrinking near the event (Figure 1).

In [ ]:
def generate_synthetic_kalshi_ladder(true_rate: float, n_days: int = 160, strike_step: float = 0.25):
    """Returns a DataFrame: one row per day-before-meeting, with mean/median/mode
    implied by a synthetic strike ladder that noisily converges to true_rate."""
    strikes = np.arange(true_rate - 2.0, true_rate + 2.0 + strike_step, strike_step)
    rows = []
    for days_before in range(n_days, -1, -1):
        noise_scale = 0.5 * (days_before / n_days) + 0.02
        center = true_rate + rng.normal(0, noise_scale)
        spread = 0.15 + 0.6 * (days_before / n_days)
        # "Yes" price of exceeding each strike ~ survival function of a logistic centered at `center`
        yes_prices = 1 / (1 + np.exp((strikes - center) / max(spread, 0.05)))
        yes_prices = np.clip(yes_prices, 0.01, 0.99)
        dist = ladder_to_pdf(list(strikes), list(yes_prices))
        rows.append({
            "days_before": days_before,
            "mean": dist.mean,
            "median": dist.median,
            "mode": dist.mode,
        })
    return pd.DataFrame(rows)


def generate_synthetic_fed_funds_futures(true_rate: float, n_days: int = 160):
    """Coarser, noisier point-forecast series to stand in for fed funds futures --
    per the FEDS paper, futures converge more slowly / less precisely than Kalshi's mode."""
    rows = []
    for days_before in range(n_days, -1, -1):
        noise_scale = 0.6 * (days_before / n_days) + 0.03
        rows.append({"days_before": days_before, "futures": true_rate + rng.normal(0, noise_scale)})
    return pd.DataFrame(rows)

## 2. Simulate several FOMC meetings and aggregate

The FEDS paper's Figure 1 averages absolute forecast error **across all FOMC meetings** in the sample, by days-before-meeting. We do the same here across a handful of synthetic meetings with different realized rates.

In [ ]:
synthetic_meetings = [4.50, 4.25, 4.25, 4.00, 3.75, 3.75]  # realized fed funds target (upper bound), per meeting

kalshi_errors = []
futures_errors = []

for true_rate in synthetic_meetings:
    ladder_df = generate_synthetic_kalshi_ladder(true_rate)
    futures_df = generate_synthetic_fed_funds_futures(true_rate)

    for col in ["mean", "median", "mode"]:
        kalshi_errors.append(pd.DataFrame({
            "days_before": ladder_df["days_before"],
            "abs_error": (ladder_df[col] - true_rate).abs(),
            "series": col,
        }))

    futures_errors.append(pd.DataFrame({
        "days_before": futures_df["days_before"],
        "abs_error": (futures_df["futures"] - true_rate).abs(),
        "series": "fed_funds_futures",
    }))

all_errors = pd.concat(kalshi_errors + futures_errors, ignore_index=True)
agg = all_errors.groupby(["series", "days_before"])["abs_error"].mean().reset_index()
agg.head()

## 3. Plot (Figure 1 style)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

colors = {"mean": "tab:blue", "median": "tab:red", "mode": "tab:green", "fed_funds_futures": "tab:purple"}
labels = {
    "mean": "Kalshi Mean Forecast Error",
    "median": "Kalshi Median Forecast Error",
    "mode": "Kalshi Mode Forecast Error",
    "fed_funds_futures": "Fed Funds Futures Forecast Error",
}

for series, df_s in agg.groupby("series"):
    df_s = df_s.sort_values("days_before")
    ax.plot(df_s["days_before"], df_s["abs_error"], label=labels[series], color=colors[series], linewidth=1.2)

ax.invert_xaxis()
ax.set_xlabel("Days before FOMC")
ax.set_ylabel("Forecast prediction error (percent)")
ax.set_title("SYNTHETIC DATA -- replace with real Kalshi pull (see src/kalshi_api.py)")
ax.legend()
plt.tight_layout()
plt.show()

## 4. Next steps to make this a real replication

1. Replace `generate_synthetic_kalshi_ladder` with a real pull: use `src/kalshi_api.py`'s `list_markets` / `get_market_candlesticks` to get the daily "Yes" price for every strike in a fed funds rate event, then run `kalshi_utils.ladder_to_pdf` on each day's ladder.
2. Replace `generate_synthetic_fed_funds_futures` with real CME fed funds futures settlement data.
3. Repeat across every FOMC meeting in your sample window (the FEDS paper uses 2022-2025) and aggregate exactly as above.
4. Overlay the NY Fed Survey of Market Expectations median-of-modal-path points (with error bars) as in the original Figure 1, once you've ingested the SME PDFs (see `paper/sections/2_replication.md`).
5. Once this matches the qualitative shape of the original Figure 1, extend the same pipeline to the *thin* series (GDP, recession probability) for `paper/sections/3_manipulation_risk.md` -- there, you want volume/open-interest, not just forecast error.